In [1]:
using Pkg
Pkg.activate("/global/homes/j/jgmorawe/desi-emulators-pipeline")

  Activating project at `~/desi-emulators-pipeline`


In [ ]:
using AbstractCosmologicalEmulators
using PyCall
using Plots
using NPZ
using DelimitedFiles
using DataInterpolations

In [ ]:
# Loads the different emulators (ln10As basis and sigma8 basis)
home_dir = "/global/homes/j/jgmorawe/desi-emulators-pipeline"
ACE_emu_ln10As_basis = AbstractCosmologicalEmulators.load_trained_emulator(home_dir * "/trained_ace_class_ln10As_basis_mnuw0wacdm_10000/")
ACE_emu_sigma8_basis = AbstractCosmologicalEmulators.load_trained_emulator(home_dir * "/trained_ace_class_sigma8_basis_mnuw0wacdm_10000/")

In [ ]:
# Picks a certain set of parameters
z, ln10As, ns, H0, ombh2, omch2, Mnu, w0, wa = 0.9, 2.8, 1.02, 59, 0.0235, 0.15, 0.06, -0.7, -0.9
emulator_input = [z, ln10As, ns, H0, ombh2, omch2, Mnu, w0, wa]

In [ ]:
# Computes the exact background quantities
classy = pyimport("classy")
np = pyimport("numpy")
CosmoDict = Dict("z" => z, "ln10As" => ln10As, "ns" => ns, "H0" => H0, "omega_b" => ombh2, "omega_cdm" => omch2, "Mnu" => Mnu, "w0" => w0, "wa" => wa)
z = CosmoDict["z"]
cosmo_params = Dict("output" => "mPk", "P_k_max_h/Mpc" => 20.0, "z_pk" => "0.0,3.", "h" => CosmoDict["H0"] / 100, "omega_b" => CosmoDict["omega_b"],
                    "omega_cdm" => CosmoDict["omega_cdm"], "ln10^{10}A_s" => CosmoDict["ln10As"], "n_s" => CosmoDict["ns"], "tau_reio" => 0.0568,
                    "N_ur" => 2.0308, "N_ncdm" => 1, "m_ncdm" => CosmoDict["Mnu"], "use_ppf" => "yes", "w0_fld" => CosmoDict["w0"], "wa_fld" => CosmoDict["wa"],
                    "fluid_equation_of_state" => "CLP", "cs2_fld" => 1.0, "Omega_Lambda" => 0.0, "Omega_scf" => 0.0)
cosmo = classy.Class()
cosmo.set(cosmo_params)
cosmo.compute()
# Computes sigma8 at z=0
sigma8 = cosmo.sigma8
CosmoDict["sigma8"] = sigma8
# Computes sigma8 at z
sigma8_z = cosmo.sigma(8.0 / (CosmoDict["H0"] / 100), z)
# Sound horizon at drag epoch
r_drag = cosmo.rs_drag
# Background quantities at redshift z
H_z = cosmo.Hubble(z) * 299792.458
r_z = cosmo.comoving_distance(z)
# Growth factor D(z) and growth rate f(z)
D_z = cosmo.scale_independent_growth_factor(z)
f_z = cosmo.scale_independent_growth_factor_f(z)
cosmo.struct_cleanup()
cosmo.empty()
# Results outputted for each training sample
result_ln10As_basis_exact = [sigma8, sigma8_z, r_drag, H_z, r_z, D_z, f_z]
result_sigma8_basis_exact = [CosmoDict["ln10As"], sigma8_z, r_drag, H_z, r_z, D_z, f_z]

In [ ]:
# Computes the emulator predictions for the same output vector
result_ln10As_basis_emu = AbstractCosmologicalEmulators.run_emulator(emulator_input, nothing, ACE_emu_ln10As_basis)
result_sigma8_basis_emu = AbstractCosmologicalEmulators.run_emulator(emulator_input, nothing, ACE_emu_sigma8_basis)

In [ ]:
# Calculates the percentage level error associated with each output vector
println("Fractional Error ln10As basis: ", (result_ln10As_basis_emu .- result_ln10As_basis_exact) ./ result_ln10As_basis_exact)
println("Fractional Error sigma8 basis: ", (result_sigma8_basis_emu .- result_sigma8_basis_exact) ./ result_sigma8_basis_exact)
# Note: the errors for the sigma8 basis are bigger, presumably this will go away with a larger training size, but please let us know if this problem persists